# Data Transformations & Tables (Bronze → Silver → Gold)


> **Integration variant:** Outputs are written only to `workspace.default.p2_team_*`. Existing `p2_*` tables are not read or overwritten. Execute stages in order: `bronze`, `silver`, `gold`, `validate`. Original business logic retained.


In [0]:
from pyspark.sql import functions as F, Window

dbutils.widgets.dropdown("stage", "explore", ["explore", "bronze", "silver", "gold", "validate", "all"])
STAGE_OVERRIDE = ""   # <-- "bronze", "silver", "gold", "validate" o "all". Déjalo "" para usar el parámetro del Job
STAGE = STAGE_OVERRIDE or dbutils.widgets.get("stage")
assert STAGE in {"explore", "bronze", "silver", "gold", "validate", "all"}, f"stage inválido: {STAGE!r}"

SRC = "/Volumes/workspace/default/applebees_datachefs_raw/OneDrive_1_5-10-2026/"
SCHEMA = "workspace.default"
TABLE_PREFIX = "p2_team_"

def table_name(name):
    assert name.startswith("p2_"), f"Unexpected table name: {name}"
    return TABLE_PREFIX + name[len("p2_"):]

def table_full_name(name):
    return f"{SCHEMA}.{table_name(name)}"

EXPECTED = {   # conteos confirmados en la guía
    "p2_bronze_checks": 2_142_846,
    "p2_bronze_employees": 156_975,
    "p2_bronze_items": 16_697,
    "p2_bronze_locations": 185,
    "p2_bronze_revenue_centers": 23,
}

# ---- Parámetros de riesgo (documentar como supuestos en el informe) ----
HIGH_DISCOUNT_PCT = 0.50   # descuento total / venta bruta del check => risk_high_discount
MIN_CHECKS_PEER   = 100    # mínimo de checks para comparar location x revenue center
MIN_ITEMS_EMP     = 100    # mínimo de items vendidos para comparar empleados
Z_LOC             = 2.0    # outliers location x revenue center (valor absoluto)
Z_EMP             = 3.0    # outliers de empleados (solo lado alto)

# ---- Helpers compartidos ----
T = lambda t: spark.table(table_full_name(t))

def sdiv(a, b):  # división segura: null si el denominador es 0
    return F.when(b != 0, a / b)

def latest_by(df, keys):  # una fila por llave (la de ingesta más reciente)
    w = Window.partitionBy(*keys).orderBy(F.col("ingest_ts").desc(), F.col("source_file").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

print("Stage:", STAGE)

Stage: gold


## Bronze: ingesta cruda + trazabilidad (`source_file`, `ingest_ts`)

In [0]:
def run_bronze():
    sources = {
        "p2_bronze_checks":          SRC + "facts/checks/",
        "p2_bronze_employees":       SRC + "dimensions/employees/",
        "p2_bronze_items":           SRC + "dimensions/items/",
        "p2_bronze_locations":       SRC + "dimensions/locations/",
        "p2_bronze_revenue_centers": SRC + "dimensions/revenue_centers/",
    }
    for table, path in sources.items():
        df = (spark.read
              .option("multiLine", "true")
              .option("recursiveFileLookup", "true")
              .option("pathGlobFilter", "*.json")
              .json(path)
              .select("*",
                      F.col("_metadata.file_path").alias("source_file"),
                      F.col("_metadata.file_name").alias("source_file_name"))
              .withColumn("ingest_ts", F.current_timestamp()))
        (df.write.mode("overwrite").option("overwriteSchema", "true")
           .saveAsTable(table_full_name(table)))
        n = T(table).count()
        print(f"{table}: {n:,} filas (esperadas {EXPECTED[table]:,}) -> "
              f"{'OK' if n == EXPECTED[table] else 'REVISAR'}")

## Silver: checks, items, descuentos, pagos, voids + flags de anomalías

In [0]:
def _cnt(c):  # tamaño de arreglo, nulo -> 0
    return F.greatest(F.size(c), F.lit(0))

def _asum(arr, expr):  # suma de un campo dentro de un arreglo
    return F.expr(f"coalesce(aggregate({arr}, 0D, (acc, x) -> acc + coalesce({expr}, 0D)), 0D)")

def _save(df, name):
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(table_full_name(name)))
    print(f"{name}: {T(name).count():,} filas")

def run_silver():
    b = T("p2_bronze_checks")

    # Clave compuesta: LocationId + BusinessDate + Id
    keyed = (b
        .withColumn("check_id", F.col("Id"))
        .withColumn("check_key", F.concat_ws("|", F.col("LocationId").cast("string"),
                                             F.col("BusinessDate"), F.col("Id").cast("string")))
        .withColumn("business_date", F.expr("try_cast(BusinessDate AS DATE)")))
    K = ["check_key", "LocationId", "check_id", "business_date"]

    # ---------- p2_silver_checks (una fila por check, sin arreglos) ----------
    item_disc = F.expr("""coalesce(aggregate(ItemsSold, 0D, (acc, x) -> acc +
        coalesce(aggregate(x.Discounts, 0D, (a, d) -> a + coalesce(d.Amount, 0D)), 0D)), 0D)""")
    w = Window.partitionBy("check_key")
    checks = (keyed.select(
            *K, "BusinessDate", "RevenueCenterId", "TableName", "LoyaltyId",
            "TrafficCount", "ServiceCharge", "OpenTime", "CloseTime",
            F.expr("try_cast(OpenTime AS TIMESTAMP)").alias("open_time_local"),
            F.expr("try_cast(CloseTime AS TIMESTAMP)").alias("close_time_local"),
            _cnt("ItemsSold").alias("item_line_count"),
            _cnt("Payments").alias("payment_count"),
            _cnt("Discounts").alias("check_discount_count"),
            _cnt("Voids").alias("void_count"),
            F.expr("coalesce(size(filter(ItemsSold, x -> coalesce(size(x.Discounts), 0) > 0)), 0)").alias("item_discount_count"),
            _asum("ItemsSold", "x.Quantity").alias("total_item_quantity"),
            _asum("ItemsSold", "x.GrossPrice").alias("total_gross_price"),
            _asum("ItemsSold", "x.SoldPrice").alias("total_sold_price"),
            item_disc.alias("item_discount_amount"),
            _asum("Discounts", "x.Amount").alias("check_discount_amount"),
            _asum("Payments", "x.Amount").alias("payment_amount"),
            _asum("Payments", "x.Gratuity").alias("gratuity_raw"),
            _asum("Tax", "x.Amount").alias("tax_amount"),
            _asum("Voids", "x.Amount").alias("void_amount"),
            "source_file", "ingest_ts")
        # Anomalías como indicadores (no se elimina nada)
        .withColumn("flag_no_items",    F.col("item_line_count") == 0)
        .withColumn("flag_no_payments", F.col("payment_count") == 0)
        .withColumn("flag_empty_check", (F.col("item_line_count") == 0) & (F.col("payment_count") == 0))
        .withColumn("flag_zero_duration",
                    F.coalesce(F.col("open_time_local") == F.col("close_time_local"), F.lit(False)))
        .withColumn("flag_negative_duration",
                    F.coalesce(F.col("close_time_local") < F.col("open_time_local"), F.lit(False)))
        .withColumn("flag_discount_both_levels",
                    (F.col("check_discount_count") > 0) & (F.col("item_discount_count") > 0))
        .withColumn("flag_duplicate_key", F.count(F.lit(1)).over(w) > 1)
        .withColumn("payment_minus_sales_tax",
                    F.round(F.col("payment_amount") - F.col("total_sold_price") - F.col("tax_amount"), 2)))
    _save(checks, "p2_silver_checks")

    # ---------- p2_silver_items_sold (una fila por línea de item) ----------
    item_fields = ["Sequence", "ParentSequence", "ItemId", "ItemName", "ItemPLU", "ItemMajorCatId",
                   "ItemMajorCatName", "ItemSubCatId", "ItemSubCatName", "EmployeeId",
                   "RevenueCenterId", "PosSourceId", "Seat", "Quantity", "GrossPrice", "SoldPrice", "OrderTime"]
    items = (keyed.select(*K, F.posexplode("ItemsSold").alias("item_pos", "it"))
        .select(*K, "item_pos", *[F.col(f"it.{f}").alias(f) for f in item_fields],
                F.expr("try_cast(it.OrderTime AS TIMESTAMP)").alias("order_time_local"),
                _cnt("it.Discounts").alias("item_discount_count"),
                _asum("it.Discounts", "x.Amount").alias("item_discount_amount")))
    _save(items, "p2_silver_items_sold")

    # ---------- p2_silver_discounts (check e item, con discount_level) ----------
    chk = (keyed.select(*K, F.explode("Discounts").alias("d"))
        .select(*K, F.lit("check").alias("discount_level"),
                F.lit(None).cast("long").alias("item_sequence"), "d.*"))
    itm = (keyed.select(*K, F.explode("ItemsSold").alias("it"))
        .select(*K, F.col("it.Sequence").alias("item_sequence"),
                F.explode("it.Discounts").alias("d"))
        .select(*K, F.lit("item").alias("discount_level"), "item_sequence", "d.*"))
    _save(chk.unionByName(itm), "p2_silver_discounts")

    # ---------- p2_silver_payments ----------
    pay = (keyed.select(*K, F.posexplode("Payments").alias("payment_pos", "p"))
           .select(*K, "payment_pos", "p.*"))
    _save(pay, "p2_silver_payments")

    # ---------- p2_silver_voids ----------
    vd = (keyed.select(*K, F.posexplode("Voids").alias("void_pos", "v"))
          .select(*K, "void_pos", "v.*"))
    _save(vd, "p2_silver_voids")

    # ---------- Validaciones rápidas ----------
    n_b = b.count(); n_s = T("p2_silver_checks").count()
    print(f"\nBronze checks: {n_b:,} | Silver checks: {n_s:,} -> {'OK' if n_b == n_s else 'REVISAR'}")
    display(T("p2_silver_checks").agg(
        *[F.sum(F.col(c).cast("int")).alias(c) for c in
          ["flag_no_items", "flag_no_payments", "flag_empty_check", "flag_zero_duration",
           "flag_negative_duration", "flag_discount_both_levels", "flag_duplicate_key"]]))

## Gold: 4 tablas de riesgo
- `p2_gold_transaction_risk_flags`: una fila por check
- `p2_gold_daily_risk_metrics`: location × día
- `p2_gold_location_risk_comparison`: location × revenue center
- `p2_gold_employee_risk_profile`: empleado × location (excluye `EmployeeId = 0` de la comparación)

In [0]:
def run_gold():
    c = T("p2_silver_checks")
    n = lambda col: F.col(col).cast("int")

    dim_location = (latest_by(T("p2_bronze_locations"), ["Id"])
        .select(F.col("Id").alias("LocationId"), F.col("Name").alias("location_name")))
    dim_rc = (latest_by(T("p2_bronze_revenue_centers"), ["Id"])
        .select(F.col("Id").alias("RevenueCenterId"), F.col("Name").alias("revenue_center_name")))

    # ---------- 1) p2_gold_transaction_risk_flags (una fila por check) ----------
    disc_total = F.coalesce("item_discount_amount", F.lit(0.0)) + F.coalesce("check_discount_amount", F.lit(0.0))
    tx = (c
      .withColumn("discount_total_amount", disc_total)
      .withColumn("discount_pct_of_gross", sdiv(F.col("discount_total_amount"), F.col("total_gross_price")))
      .withColumn("risk_fully_voided", F.coalesce(F.col("flag_no_items") & (F.col("void_count") > 0), F.lit(False)))
      .withColumn("risk_high_discount", F.coalesce(F.col("discount_pct_of_gross") >= HIGH_DISCOUNT_PCT, F.lit(False)))
      .withColumn("risk_no_payment", F.coalesce((F.col("item_line_count") > 0) & (F.col("payment_count") == 0), F.lit(False)))
      .withColumn("risk_discount_both_levels", F.coalesce(F.col("flag_discount_both_levels"), F.lit(False)))
      .withColumn("risk_score",
          n("risk_fully_voided") + n("risk_high_discount") + n("risk_no_payment") + n("risk_discount_both_levels"))
      .join(dim_location, "LocationId", "left")
      .join(dim_rc, "RevenueCenterId", "left")
      .select("check_key", "LocationId", "location_name", "RevenueCenterId", "revenue_center_name",
              "business_date", "total_gross_price", "total_sold_price", "payment_amount",
              "void_count", "void_amount", "item_discount_amount", "check_discount_amount",
              "discount_total_amount", "discount_pct_of_gross",
              "risk_fully_voided", "risk_high_discount", "risk_no_payment",
              "risk_discount_both_levels", "risk_score", "flag_duplicate_key"))
    _save(tx, "p2_gold_transaction_risk_flags")
    tx = T("p2_gold_transaction_risk_flags")   # las tablas 2 y 3 salen de lo ya guardado

    # ---------- 2) p2_gold_daily_risk_metrics (location x día) ----------
    daily = (tx.groupBy("LocationId", "location_name", "business_date").agg(
        F.count("*").alias("checks"),
        F.sum("total_gross_price").alias("gross_sales"),
        F.sum("total_sold_price").alias("sold_sales"),
        F.sum("item_discount_amount").alias("item_discount_amount"),
        F.sum("check_discount_amount").alias("check_discount_amount"),
        F.sum("void_amount").alias("void_amount"),
        F.avg(n("risk_fully_voided")).alias("rate_fully_voided"),
        F.avg(n("risk_high_discount")).alias("rate_high_discount"),
        F.avg(n("risk_no_payment")).alias("rate_no_payment"),
        F.avg(n("risk_discount_both_levels")).alias("rate_discount_both_levels"),
        F.avg((F.col("void_count") > 0).cast("int")).alias("rate_checks_with_void"))
      .withColumn("discount_pct", sdiv(F.col("item_discount_amount") + F.col("check_discount_amount"), F.col("gross_sales")))
      .withColumn("void_pct", sdiv(F.col("void_amount"), F.col("gross_sales"))))
    _save(daily, "p2_gold_daily_risk_metrics")

    # ---------- 3) p2_gold_location_risk_comparison (location x revenue center) ----------
    lrc = (tx.groupBy("LocationId", "location_name", "RevenueCenterId", "revenue_center_name").agg(
        F.count("*").alias("checks"),
        F.sum("total_gross_price").alias("gross_sales"),
        F.sum("void_amount").alias("void_amount"),
        (F.sum("item_discount_amount") + F.sum("check_discount_amount")).alias("discount_amount"),
        F.avg(n("risk_fully_voided")).alias("rate_fully_voided"))
      .withColumn("void_pct", sdiv(F.col("void_amount"), F.col("gross_sales")))
      .withColumn("discount_pct", sdiv(F.col("discount_amount"), F.col("gross_sales"))))

    w_rc = Window.partitionBy("RevenueCenterId")
    stats = (lrc.filter(F.col("checks") >= MIN_CHECKS_PEER)
      .withColumn("void_pct_peer_avg", F.avg("void_pct").over(w_rc))
      .withColumn("void_pct_peer_std", F.stddev("void_pct").over(w_rc))
      .withColumn("discount_pct_peer_avg", F.avg("discount_pct").over(w_rc))
      .withColumn("discount_pct_peer_std", F.stddev("discount_pct").over(w_rc))
      .withColumn("void_z", sdiv(F.col("void_pct") - F.col("void_pct_peer_avg"), F.col("void_pct_peer_std")))
      .withColumn("discount_z", sdiv(F.col("discount_pct") - F.col("discount_pct_peer_avg"), F.col("discount_pct_peer_std")))
      .withColumn("is_outlier", (F.abs("void_z") >= Z_LOC) | (F.abs("discount_z") >= Z_LOC)))
    loc_cmp = lrc.join(stats.select("LocationId", "RevenueCenterId", "void_z", "discount_z", "is_outlier"),
                       ["LocationId", "RevenueCenterId"], "left")
    _save(loc_cmp, "p2_gold_location_risk_comparison")

    # ---------- 4) p2_gold_employee_risk_profile (empleado x location) ----------
    K = ["EmployeeId", "LocationId"]
    dim_employee = (latest_by(T("p2_bronze_employees"), ["Id"])
        .select(F.col("Id").alias("EmployeeId"), F.col("Name").alias("employee_name"),
                F.col("Deleted").alias("employee_deleted")))

    sold = (T("p2_silver_items_sold").groupBy(K).agg(
        F.count("*").alias("items_sold_lines"),
        F.sum("GrossPrice").alias("gross_sales"),
        F.countDistinct("check_key").alias("checks_touched")))

    voids = (T("p2_silver_voids").groupBy(K).agg(
        F.count("*").alias("void_count"),
        F.sum("Amount").alias("void_amount"),
        F.sum(F.col("ManagerId").isNotNull().cast("int")).alias("voids_with_manager"),
        F.countDistinct("check_key").alias("checks_with_void")))

    disc = T("p2_silver_discounts")
    discounts = (disc.groupBy(K).agg(
        F.sum((F.col("discount_level") == "check").cast("int")).alias("check_discount_count"),
        F.sum(F.when(F.col("discount_level") == "check", F.col("Amount"))).alias("check_discount_amount"),
        F.sum((F.col("discount_level") == "item").cast("int")).alias("item_discount_count"),
        F.sum(F.when(F.col("discount_level") == "item", F.col("Amount"))).alias("item_discount_amount"),
        F.sum(F.col("ManagerId").isNotNull().cast("int")).alias("discounts_with_manager")))

    emp = (sold
      .join(voids, K, "full_outer")
      .join(discounts, K, "full_outer")
      .na.fill(0, ["items_sold_lines", "gross_sales", "checks_touched", "void_count", "void_amount",
                   "voids_with_manager", "checks_with_void", "check_discount_count", "check_discount_amount",
                   "item_discount_count", "item_discount_amount", "discounts_with_manager"])
      .withColumn("discount_amount_total", F.col("check_discount_amount") + F.col("item_discount_amount"))
      .withColumn("void_rate_per_item", sdiv(F.col("void_count"), F.col("items_sold_lines")))
      .withColumn("void_pct_of_sales", sdiv(F.col("void_amount"), F.col("gross_sales")))
      .withColumn("discount_pct_of_sales", sdiv(F.col("discount_amount_total"), F.col("gross_sales")))
      .withColumn("voids_no_manager_pct", sdiv(F.col("void_count") - F.col("voids_with_manager"), F.col("void_count")))
      .join(dim_employee, "EmployeeId", "left")
      .join(dim_location, "LocationId", "left"))

    # Comparación contra pares del mismo location (EmployeeId 0 = placeholder del POS, se excluye)
    peers = emp.filter((F.col("items_sold_lines") >= MIN_ITEMS_EMP) & (F.col("EmployeeId") != 0))
    w_loc = Window.partitionBy("LocationId")
    z = (peers
      .withColumn("void_avg", F.avg("void_rate_per_item").over(w_loc))
      .withColumn("void_std", F.stddev("void_rate_per_item").over(w_loc))
      .withColumn("disc_avg", F.avg("discount_pct_of_sales").over(w_loc))
      .withColumn("disc_std", F.stddev("discount_pct_of_sales").over(w_loc))
      .withColumn("void_z", sdiv(F.col("void_rate_per_item") - F.col("void_avg"), F.col("void_std")))
      .withColumn("discount_z", sdiv(F.col("discount_pct_of_sales") - F.col("disc_avg"), F.col("disc_std")))
      .withColumn("is_outlier", (F.col("void_z") >= Z_EMP) | (F.col("discount_z") >= Z_EMP))
      .select(*K, "void_z", "discount_z", "is_outlier"))

    emp_final = (emp.join(z, K, "left")
      .select("EmployeeId", "employee_name", "employee_deleted", "LocationId", "location_name",
              "checks_touched", "items_sold_lines", "gross_sales",
              "void_count", "void_amount", "void_rate_per_item", "void_pct_of_sales",
              "voids_with_manager", "voids_no_manager_pct", "checks_with_void",
              "check_discount_count", "check_discount_amount",
              "item_discount_count", "item_discount_amount",
              "discount_amount_total", "discount_pct_of_sales", "discounts_with_manager",
              "void_z", "discount_z", "is_outlier"))
    _save(emp_final, "p2_gold_employee_risk_profile")

## Validación (evidencia): conciliaciones Silver y Gold

In [0]:
def run_validation():
    """Conciliaciones de Silver y Gold. Es la evidencia que pide la guía; no escribe tablas."""
    c = T("p2_silver_checks")
    def _sum(tbl, col, cond=None):
        d = T(tbl)
        if cond: d = d.filter(cond)
        return d.agg(F.sum(col)).first()[0] or 0.0

    # --- Silver: la suma del detalle debe igualar la de p2_silver_checks ---
    pairs = [
        ("Items: SoldPrice",    _sum("p2_silver_items_sold", "SoldPrice"),  _sum("p2_silver_checks", "total_sold_price")),
        ("Pagos: Amount",       _sum("p2_silver_payments", "Amount"),       _sum("p2_silver_checks", "payment_amount")),
        ("Voids: Amount",       _sum("p2_silver_voids", "Amount"),          _sum("p2_silver_checks", "void_amount")),
        ("Desc. check: Amount", _sum("p2_silver_discounts", "Amount", "discount_level = 'check'"), _sum("p2_silver_checks", "check_discount_amount")),
        ("Desc. item: Amount",  _sum("p2_silver_discounts", "Amount", "discount_level = 'item'"),  _sum("p2_silver_checks", "item_discount_amount")),
    ]
    print("CONCILIACIÓN SILVER: detalle vs. checks")
    for name, detail, header in pairs:
        print(f"{name:22s} detalle={detail:>16,.2f}  checks={header:>16,.2f}  -> {'OK' if abs(detail-header) < 0.01 else 'REVISAR'}")

    print("\nCHECKS SIN ITEMS PERO CON PAGO")
    display(c.filter("flag_no_items AND NOT flag_no_payments").agg(
        F.count("*").alias("checks"),
        F.sum((F.col("void_count") > 0).cast("int")).alias("con_voids"),
        F.round(F.sum("payment_amount"), 2).alias("monto_pagado"),
        F.round(F.avg("payment_amount"), 2).alias("pago_promedio")))

    print("\nDESCUENTO EN AMBOS NIVELES")
    display(c.filter("flag_discount_both_levels").agg(
        F.count("*").alias("checks"),
        F.round(F.sum("check_discount_amount"), 2).alias("desc_check_total"),
        F.round(F.sum("item_discount_amount"), 2).alias("desc_item_total"),
        F.sum((F.abs(F.col("check_discount_amount") - F.col("item_discount_amount")) < 0.01).cast("int")).alias("montos_iguales")))

    # --- Gold: contra Silver y entre sí ---
    tx, dm = T("p2_gold_transaction_risk_flags"), T("p2_gold_daily_risk_metrics")
    lc, ep = T("p2_gold_location_risk_comparison"), T("p2_gold_employee_risk_profile")
    s = lambda df, col: df.agg(F.sum(col)).first()[0] or 0.0
    ok = lambda cond: "OK" if cond else "REVISAR"

    n_s = c.count()
    cnts = [("tx", tx.count()), ("daily", int(s(dm, "checks"))), ("loc x rc", int(s(lc, "checks")))]
    print("\nCONCILIACIÓN GOLD")
    print(f"checks silver={n_s:,} | " + " | ".join(f"{k}={v:,}" for k, v in cnts),
          "->", ok(all(v == n_s for _, v in cnts)))
    g_silver = s(c, "total_gross_price")
    for k, v in [("tx", s(tx, "total_gross_price")), ("daily", s(dm, "gross_sales")), ("loc x rc", s(lc, "gross_sales"))]:
        print(f"ventas brutas {k:9s} {v:>16,.2f} vs silver {g_silver:>16,.2f} -> {ok(abs(v - g_silver) < 0.01)}")
    v_silver = s(T("p2_silver_voids"), "Amount")
    print(f"void $ empleados={s(ep, 'void_amount'):,.2f} vs silver={v_silver:,.2f} -> {ok(abs(s(ep, 'void_amount') - v_silver) < 0.01)}")
    print(f"void n empleados={int(s(ep, 'void_count')):,} vs silver={T('p2_silver_voids').count():,} -> "
          f"{ok(int(s(ep, 'void_count')) == T('p2_silver_voids').count())}")
    d_n = int(s(ep, "check_discount_count") + s(ep, "item_discount_count"))
    print(f"desc n empleados={d_n:,} vs silver={T('p2_silver_discounts').count():,} -> {ok(d_n == T('p2_silver_discounts').count())}")
    d_amt = s(ep, "discount_amount_total"); d_silver = s(T("p2_silver_discounts"), "Amount")
    print(f"desc $ empleados={d_amt:,.2f} vs silver={d_silver:,.2f} -> {ok(abs(d_amt - d_silver) < 0.01)}")

    print("\nLLAVES ÚNICAS / NULOS")
    print("daily dup:", dm.count() - dm.select("LocationId", "business_date").distinct().count())
    print("loc x rc dup:", lc.count() - lc.select("LocationId", "RevenueCenterId").distinct().count())
    print("emp dup:", ep.count() - ep.select("EmployeeId", "LocationId").distinct().count())
    print("tx sin location_name:", tx.filter("location_name IS NULL").count(),
          "| sin revenue_center_name:", tx.filter("revenue_center_name IS NULL").count(),
          "| flags nulos:", tx.filter("risk_score IS NULL").count())
    print("empleados sin nombre:", ep.filter("employee_name IS NULL").count(),
          "| de ellos con id != 0:", ep.filter("employee_name IS NULL AND EmployeeId != 0").count())

    print("\nDISTRIBUCIÓN risk_score")
    tx.groupBy("risk_score").count().orderBy("risk_score").show()
    print("outliers locations:", lc.filter("is_outlier").count(), "de", lc.filter("is_outlier IS NOT NULL").count(),
          "| outliers empleados:", ep.filter("is_outlier").count(), "de", ep.filter("is_outlier IS NOT NULL").count())
    print("rango de fechas:", dm.agg(F.min("business_date"), F.max("business_date")).first())

## Despacho según `STAGE`

In [0]:
if STAGE in ("bronze", "all"):   run_bronze()
if STAGE in ("silver", "all"):   run_silver()
if STAGE in ("gold",   "all"):   run_gold()
if STAGE in ("validate", "all"): run_validation()

p2_gold_transaction_risk_flags: 2,142,846 filas
p2_gold_daily_risk_metrics: 14,585 filas
p2_gold_location_risk_comparison: 1,026 filas
p2_gold_employee_risk_profile: 3,226 filas


In [0]:
print("=== TEAM BRONZE VERIFICATION ===")

names = [
    "checks",
    "employees",
    "items",
    "locations",
    "revenue_centers"
]

for name in names:
    table = f"workspace.default.p2_team_bronze_{name}"

    try:
        n = spark.table(table).count()
        print(f"PASS | {table} | {n:,} rows")
    except Exception as e:
        print(f"FAILED | {table}")
        print(str(e).splitlines()[0])

=== TEAM BRONZE VERIFICATION ===
PASS | workspace.default.p2_team_bronze_checks | 2,142,846 rows
PASS | workspace.default.p2_team_bronze_employees | 156,975 rows
PASS | workspace.default.p2_team_bronze_items | 16,697 rows
PASS | workspace.default.p2_team_bronze_locations | 185 rows
PASS | workspace.default.p2_team_bronze_revenue_centers | 23 rows


In [0]:
print("=== TEAM SILVER VERIFICATION ===")

expected = {
    "checks": 2142846,
    "items_sold": 28558925,
    "discounts": 516690,
    "payments": 2232092,
    "voids": 178303
}

for name, expected_count in expected.items():
    table = f"workspace.default.p2_team_silver_{name}"

    actual = spark.table(table).count()
    status = "PASS" if actual == expected_count else "REVIEW"

    print(
        f"{status} | {table} | "
        f"{actual:,} rows | Expected: {expected_count:,}"
    )

    assert actual == expected_count, (
        f"Count mismatch in {table}"
    )

print("TEAM SILVER VERIFICATION: PASSED")

=== TEAM SILVER VERIFICATION ===
PASS | workspace.default.p2_team_silver_checks | 2,142,846 rows | Expected: 2,142,846
PASS | workspace.default.p2_team_silver_items_sold | 28,558,925 rows | Expected: 28,558,925
PASS | workspace.default.p2_team_silver_discounts | 516,690 rows | Expected: 516,690
PASS | workspace.default.p2_team_silver_payments | 2,232,092 rows | Expected: 2,232,092
PASS | workspace.default.p2_team_silver_voids | 178,303 rows | Expected: 178,303
TEAM SILVER VERIFICATION: PASSED


In [0]:
print("=== TEAM GOLD VERIFICATION ===")

expected = {
    "transaction_risk_flags": 2142846,
    "daily_risk_metrics": 14585,
    "location_risk_comparison": 1026,
    "employee_risk_profile": 3226
}

for name, expected_count in expected.items():
    table = f"workspace.default.p2_team_gold_{name}"

    actual = spark.table(table).count()
    status = "PASS" if actual == expected_count else "REVIEW"

    print(
        f"{status} | {table} | "
        f"{actual:,} rows | Expected: {expected_count:,}"
    )

    assert actual == expected_count, (
        f"Count mismatch in {table}"
    )

print("TEAM GOLD VERIFICATION: PASSED")

=== TEAM GOLD VERIFICATION ===
PASS | workspace.default.p2_team_gold_transaction_risk_flags | 2,142,846 rows | Expected: 2,142,846
PASS | workspace.default.p2_team_gold_daily_risk_metrics | 14,585 rows | Expected: 14,585
PASS | workspace.default.p2_team_gold_location_risk_comparison | 1,026 rows | Expected: 1,026
PASS | workspace.default.p2_team_gold_employee_risk_profile | 3,226 rows | Expected: 3,226
TEAM GOLD VERIFICATION: PASSED


In [0]:
print("=== BASELINE FOR IDEMPOTENCY TEST ===")

tables = {
    "bronze": [
        "checks",
        "employees",
        "items",
        "locations",
        "revenue_centers"
    ],
    "silver": [
        "checks",
        "items_sold",
        "discounts",
        "payments",
        "voids"
    ],
    "gold": [
        "transaction_risk_flags",
        "daily_risk_metrics",
        "location_risk_comparison",
        "employee_risk_profile"
    ]
}

baseline = {}

for layer, names in tables.items():
    for name in names:
        table = f"workspace.default.p2_team_{layer}_{name}"
        count = spark.table(table).count()
        baseline[table] = count
        print(f"{table}: {count:,}")

print(f"\nTOTAL TABLES VERIFIED: {len(baseline)}")

=== BASELINE FOR IDEMPOTENCY TEST ===
workspace.default.p2_team_bronze_checks: 2,142,846
workspace.default.p2_team_bronze_employees: 156,975
workspace.default.p2_team_bronze_items: 16,697
workspace.default.p2_team_bronze_locations: 185
workspace.default.p2_team_bronze_revenue_centers: 23
workspace.default.p2_team_silver_checks: 2,142,846
workspace.default.p2_team_silver_items_sold: 28,558,925
workspace.default.p2_team_silver_discounts: 516,690
workspace.default.p2_team_silver_payments: 2,232,092
workspace.default.p2_team_silver_voids: 178,303
workspace.default.p2_team_gold_transaction_risk_flags: 2,142,846
workspace.default.p2_team_gold_daily_risk_metrics: 14,585
workspace.default.p2_team_gold_location_risk_comparison: 1,026
workspace.default.p2_team_gold_employee_risk_profile: 3,226

TOTAL TABLES VERIFIED: 14


In [0]:
print("=== IDEMPOTENCY TEST: SECOND RUN ===")

expected = {
    "bronze_checks": 2142846,
    "bronze_employees": 156975,
    "bronze_items": 16697,
    "bronze_locations": 185,
    "bronze_revenue_centers": 23,
    "silver_checks": 2142846,
    "silver_items_sold": 28558925,
    "silver_discounts": 516690,
    "silver_payments": 2232092,
    "silver_voids": 178303,
    "gold_transaction_risk_flags": 2142846,
    "gold_daily_risk_metrics": 14585,
    "gold_location_risk_comparison": 1026,
    "gold_employee_risk_profile": 3226
}

passed = 0

for name, baseline in expected.items():
    table = f"workspace.default.p2_team_{name}"
    current = spark.table(table).count()

    status = "PASS" if current == baseline else "FAIL"

    print(
        f"{status} | {name} | "
        f"Baseline: {baseline:,} | Current: {current:,}"
    )

    if current == baseline:
        passed += 1

print(f"\nRESULT: {passed}/{len(expected)} tables unchanged")

if passed == len(expected):
    print("ROW COUNT STABILITY TEST: PASSED")
else:
    print("ROW COUNT STABILITY TEST: FAILED")

=== IDEMPOTENCY TEST: SECOND RUN ===
PASS | bronze_checks | Baseline: 2,142,846 | Current: 2,142,846
PASS | bronze_employees | Baseline: 156,975 | Current: 156,975
PASS | bronze_items | Baseline: 16,697 | Current: 16,697
PASS | bronze_locations | Baseline: 185 | Current: 185
PASS | bronze_revenue_centers | Baseline: 23 | Current: 23
PASS | silver_checks | Baseline: 2,142,846 | Current: 2,142,846
PASS | silver_items_sold | Baseline: 28,558,925 | Current: 28,558,925
PASS | silver_discounts | Baseline: 516,690 | Current: 516,690
PASS | silver_payments | Baseline: 2,232,092 | Current: 2,232,092
PASS | silver_voids | Baseline: 178,303 | Current: 178,303
PASS | gold_transaction_risk_flags | Baseline: 2,142,846 | Current: 2,142,846
PASS | gold_daily_risk_metrics | Baseline: 14,585 | Current: 14,585
PASS | gold_location_risk_comparison | Baseline: 1,026 | Current: 1,026
PASS | gold_employee_risk_profile | Baseline: 3,226 | Current: 3,226

RESULT: 14/14 tables unchanged
ROW COUNT STABILITY TEST

In [0]:
from pyspark.sql import functions as F

print("=== CHECK TRANSACTION KEY VALIDATION ===")

tables = [
    "workspace.default.p2_team_bronze_checks",
    "workspace.default.p2_team_silver_checks",
    "workspace.default.p2_team_gold_transaction_risk_flags"
]

key_columns = ["LocationId", "BusinessDate", "Id"]

for table in tables:
    print(f"\nTABLE: {table}")

    df = spark.table(table)

    # Check required columns
    missing = [c for c in key_columns if c not in df.columns]

    if missing:
        print(f"REVIEW | Missing columns: {missing}")
        continue

    # Check null keys
    null_keys = df.filter(
        F.col("LocationId").isNull() |
        F.col("BusinessDate").isNull() |
        F.col("Id").isNull()
    ).count()

    # Check duplicate composite keys
    duplicate_groups = (
        df.groupBy(*key_columns)
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    status = (
        "PASS"
        if null_keys == 0 and duplicate_groups == 0
        else "REVIEW"
    )

    print(f"Null composite keys: {null_keys:,}")
    print(f"Duplicate key groups: {duplicate_groups:,}")
    print(f"STATUS: {status}")

print("\n=== VALIDATION COMPLETED ===")

=== CHECK TRANSACTION KEY VALIDATION ===

TABLE: workspace.default.p2_team_bronze_checks
Null composite keys: 0
Duplicate key groups: 0
STATUS: PASS

TABLE: workspace.default.p2_team_silver_checks
REVIEW | Missing columns: ['Id']

TABLE: workspace.default.p2_team_gold_transaction_risk_flags
REVIEW | Missing columns: ['BusinessDate', 'Id']

=== VALIDATION COMPLETED ===


In [0]:
print("=== CHECK TABLE SCHEMA INSPECTION ===")

tables = [
    "workspace.default.p2_team_bronze_checks",
    "workspace.default.p2_team_silver_checks",
    "workspace.default.p2_team_gold_transaction_risk_flags"
]

for table in tables:
    print(f"\nTABLE: {table}")

    df = spark.table(table)

    for field in df.schema.fields:
        print(f"  {field.name}: {field.dataType.simpleString()}")

print("\n=== SCHEMA INSPECTION COMPLETED ===")

=== CHECK TABLE SCHEMA INSPECTION ===

TABLE: workspace.default.p2_team_bronze_checks
  BusinessDate: string
  CloseTime: string
  Discounts: array<struct<Amount:double,DiscountId:bigint,DiscountName:string,EmployeeId:bigint,ManagerId:bigint>>
  Id: bigint
  ItemsSold: array<struct<Discounts:array<struct<Amount:double,DiscountId:bigint,DiscountName:string,EmployeeId:bigint,ManagerId:bigint>>,EmployeeId:bigint,GrossPrice:double,ItemId:bigint,ItemMajorCatId:bigint,ItemMajorCatName:string,ItemName:string,ItemPLU:bigint,ItemSubCatId:bigint,ItemSubCatName:string,OrderTime:string,ParentSequence:bigint,PosSourceId:bigint,Quantity:double,RevenueCenterId:bigint,Seat:bigint,Sequence:bigint,SoldPrice:double>>
  LocationId: bigint
  LoyaltyId: string
  OpenTime: string
  Payments: array<struct<Amount:double,EmployeeId:bigint,Gratuity:double,IsCashDebit:boolean,IsCreditCard:boolean,IsGiftCard:boolean,IsGiftCert:boolean,IsNonPOS:boolean,IsPettyCash:boolean,PayCatName:string,PayCatNo:bigint,PaymentCa

In [0]:
from pyspark.sql import functions as F

print("=== SILVER & GOLD KEY INTEGRITY VALIDATION ===")

silver = spark.table(
    "workspace.default.p2_team_silver_checks"
)

gold = spark.table(
    "workspace.default.p2_team_gold_transaction_risk_flags"
)

def validate_keys(df, name):
    print(f"\n--- {name} ---")

    total = df.count()

    null_keys = df.filter(
        F.col("check_key").isNull()
    ).count()

    duplicate_groups = (
        df.groupBy("check_key")
          .count()
          .filter(F.col("count") > 1)
          .count()
    )

    flagged_duplicates = df.filter(
        F.col("flag_duplicate_key") == True
    ).count()

    print(f"Total rows: {total:,}")
    print(f"Null check_key: {null_keys:,}")
    print(f"Duplicate key groups: {duplicate_groups:,}")
    print(f"Flagged duplicates: {flagged_duplicates:,}")

    status = (
        "PASS"
        if null_keys == 0 and duplicate_groups == 0
        else "REVIEW"
    )

    print(f"STATUS: {status}")

validate_keys(silver, "SILVER")
validate_keys(gold, "GOLD")

print("\n--- CROSS-LAYER KEY COMPARISON ---")

silver_keys = silver.select("check_key").distinct()
gold_keys = gold.select("check_key").distinct()

missing_in_gold = silver_keys.join(
    gold_keys,
    on="check_key",
    how="left_anti"
).count()

extra_in_gold = gold_keys.join(
    silver_keys,
    on="check_key",
    how="left_anti"
).count()

print(f"Silver keys missing in Gold: {missing_in_gold:,}")
print(f"Gold keys missing in Silver: {extra_in_gold:,}")

if missing_in_gold == 0 and extra_in_gold == 0:
    print("CROSS-LAYER KEY VALIDATION: PASSED")
else:
    print("CROSS-LAYER KEY VALIDATION: REVIEW")

print("\n=== VALIDATION COMPLETED ===")

=== SILVER & GOLD KEY INTEGRITY VALIDATION ===

--- SILVER ---
Total rows: 2,142,846
Null check_key: 0
Duplicate key groups: 0
Flagged duplicates: 0
STATUS: PASS

--- GOLD ---
Total rows: 2,142,846
Null check_key: 0
Duplicate key groups: 0
Flagged duplicates: 0
STATUS: PASS

--- CROSS-LAYER KEY COMPARISON ---
Silver keys missing in Gold: 0
Gold keys missing in Silver: 0
CROSS-LAYER KEY VALIDATION: PASSED

=== VALIDATION COMPLETED ===


In [0]:
from pyspark.sql import functions as F

print("=== SILVER VS GOLD FINANCIAL CONSISTENCY ===")

silver = spark.table(
    "workspace.default.p2_team_silver_checks"
)

gold = spark.table(
    "workspace.default.p2_team_gold_transaction_risk_flags"
)

financial_columns = [
    "total_gross_price",
    "total_sold_price",
    "payment_amount",
    "void_amount"
]

s = silver.select(
    "check_key",
    *[F.col(c).alias(f"s_{c}") for c in financial_columns]
)

g = gold.select(
    "check_key",
    *[F.col(c).alias(f"g_{c}") for c in financial_columns]
)

joined = s.join(g, on="check_key", how="inner")

tolerance = 0.01

for column in financial_columns:

    a = F.col(f"s_{column}")
    b = F.col(f"g_{column}")

    differences = joined.filter(
        ~a.eqNullSafe(b)
        & (
            a.isNull()
            | b.isNull()
            | (F.abs(a - b) > tolerance)
            | F.isnan(a)
            | F.isnan(b)
        )
    ).count()

    status = "PASS" if differences == 0 else "REVIEW"

    print(
        f"{status} | {column} | "
        f"Different records: {differences:,}"
    )

print("\n=== FINANCIAL VALIDATION COMPLETED ===")

=== SILVER VS GOLD FINANCIAL CONSISTENCY ===
PASS | total_gross_price | Different records: 0
PASS | total_sold_price | Different records: 0
PASS | payment_amount | Different records: 0
PASS | void_amount | Different records: 0

=== FINANCIAL VALIDATION COMPLETED ===


In [0]:
%sql
GRANT SELECT
ON TABLE workspace.default.p2_team_silver_checks
TO `A01424422@tec.mx`;

In [0]:
%sql
-- Bronze: cinco tablas
GRANT SELECT ON TABLE workspace.default.p2_team_bronze_checks
TO `A01424422@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_employees
TO `A01424422@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_items
TO `A01424422@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_locations
TO `A01424422@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_revenue_centers
TO `A01424422@tec.mx`;

-- Silver: cuatro tablas pendientes
GRANT SELECT ON TABLE workspace.default.p2_team_silver_items_sold
TO `A01424422@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_silver_discounts
TO `A01424422@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_silver_payments
TO `A01424422@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_silver_voids
TO `A01424422@tec.mx`;

In [0]:
%sql
-- Bronze: cinco tablas
GRANT SELECT ON TABLE workspace.default.p2_team_bronze_checks
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_employees
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_items
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_locations
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_bronze_revenue_centers
TO `A01739085@tec.mx`;

-- Silver: cuatro tablas pendientes
GRANT SELECT ON TABLE workspace.default.p2_team_silver_items_sold
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_silver_discounts
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_silver_payments
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_silver_voids
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_gold_transaction_risk_flags
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_gold_daily_risk_metrics
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_gold_location_risk_comparison
TO `A01739085@tec.mx`;

GRANT SELECT ON TABLE workspace.default.p2_team_gold_employee_risk_profile
TO `A01739085@tec.mx`;